# Notebook 03 — Group-stratifizierter Train/Val/Test-Split

**Ziel:** `data/splits/{train,val,test}.csv` erzeugen.

**Strategie:**
- Disk ist Ground Truth: nur Alben mit vorhandenem Cover werden berücksichtigt.
- Split-Einheit ist der **Artist**, nicht das Album: alle Alben eines Artists
  landen im selben Split → kein Datenleck durch visuelle Ähnlichkeit.
- **Stratifizierung** pro Genre: Artists werden innerhalb jedes Genres
  70 / 15 / 15 aufgeteilt.
- **SEED = 42** für Reproduzierbarkeit.

**Invariante (assert):** Kein `artist_id` darf in mehr als einem Split erscheinen.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT       = Path.cwd().parent
COVERS_DIR = ROOT / "data" / "covers"
SPLITS_DIR = ROOT / "data" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

print(f"ROOT:       {ROOT}")
print(f"COVERS_DIR: {COVERS_DIR}")
print(f"SPLITS_DIR: {SPLITS_DIR}")

## 1. Disk-Scan — Cover als Ground Truth

In [ ]:
rows = []
for jpg in sorted(COVERS_DIR.rglob("*.jpg")):
    # Dateiname: {album_id}.jpg  ODER  {artist_name}_{album_id}.jpg
    # album_id ist immer der letzte _-separierte Teil (Spotify Base62, kein Unterstrich)
    album_id = jpg.stem.rsplit("_", 1)[-1]
    rows.append({
        "album_id":   album_id,
        "genre":      jpg.parent.name,
        "cover_path": str(jpg),
    })

disk_df = pd.DataFrame(rows)
print(f"Cover auf Disk: {len(disk_df)}")
print(disk_df["genre"].value_counts().sort_index().to_string())

## 2. Metadaten joinen (artist_id aus albums_raw.csv)

In [ ]:
raw = pd.read_csv(ROOT / "data" / "albums_raw.csv")
raw = raw.drop_duplicates(subset="album_id", keep="first")

df = disk_df.merge(
    raw[["album_id", "artist_id"]],
    on="album_id",
    how="left",
)

missing_artist = df["artist_id"].isna().sum()
print(f"Alben ohne artist_id-Match: {missing_artist}")
if missing_artist > 0:
    print("WARNUNG: Diese Alben werden aus dem Split ausgeschlossen.")
    df = df.dropna(subset=["artist_id"]).reset_index(drop=True)

print(f"Alben für Split: {len(df)}")

## 3. Group-stratifizierter Split 70 / 15 / 15 nach artist_id

In [ ]:
rng = np.random.default_rng(SEED)
df["split"] = None

for genre, group in df.groupby("genre"):
    artists = group["artist_id"].unique()
    rng.shuffle(artists)
    n       = len(artists)
    n_train = int(0.70 * n)
    n_val   = int(0.15 * n)
    # Rest geht in Test (mindestens 1 Artist)

    train_artists = set(artists[:n_train])
    val_artists   = set(artists[n_train : n_train + n_val])
    test_artists  = set(artists[n_train + n_val :])

    mask = df["genre"] == genre
    df.loc[mask & df["artist_id"].isin(train_artists), "split"] = "train"
    df.loc[mask & df["artist_id"].isin(val_artists),   "split"] = "val"
    df.loc[mask & df["artist_id"].isin(test_artists),  "split"] = "test"

# Übersicht
print("\nAlben pro Split:")
print(df["split"].value_counts().to_string())
print("\nAlben pro Genre × Split:")
print(df.groupby(["genre", "split"]).size().unstack(fill_value=0).to_string())

## 4. Invariante: kein Artist in mehr als einem Split

In [ ]:
violations = 0
for artist, sub in df.groupby("artist_id"):
    splits_seen = sub["split"].unique()
    if len(splits_seen) > 1:
        print(f"VERLETZUNG: artist {artist} in {splits_seen}")
        violations += 1

assert violations == 0, f"{violations} Artist(s) in mehreren Splits — Datenleck!"
print(f"Invariante OK: 0 Verletzungen.")

## 5. CSVs schreiben

In [ ]:
cols = ["album_id", "genre", "artist_id", "cover_path"]
for split in ["train", "val", "test"]:
    out_path = SPLITS_DIR / f"{split}.csv"
    df[df["split"] == split][cols].to_csv(out_path, index=False)
    n = len(df[df["split"] == split])
    print(f"  {split}.csv  →  {n} Alben  →  {out_path}")

print("\nFertig.")